# Week 1: Research Question and Provisional Lane

## My lane: CTR / Engagement Opportunity Scoring

I choose the predefined search-intelligence lane that finds visible pages that may under-capture clicks and recommends metadata or content review. This lane is worth pursuing because CTR is directly observable in Search Console, the review action is concrete, and the cost of a bad recommendation is manageable when a human reviews the page before changing anything.

## 1) The question

**Research question:** Among visible content items, which client-content-month rows show the strongest evidence of under-captured clicks relative to their search position and demand?

- **Decision:** Which pages should enter the next SEO review queue?
- **Unit of analysis:** One pseudonymized client-content-month row, aggregated from daily Search Console and Analytics observations.
- **Output:** A transparent opportunity score, one reason code, and an action label.
- **Action:** A reviewer checks the page title, description, SERP context, intent, and content before deciding whether to revise metadata or content.
- **Cost of a wrong call:** Review time is wasted, or a team changes a page that was not actually under-capturing clicks. A severe wrong call could reduce useful traffic, so the score must not publish changes automatically.

## 2) Quick look at the real data

I use the March 2026 warehouse partition as a mid-panel feature window. June is not used here because the dataset README identifies the `_sample` month as the latest month and future labels should be kept separate from development.

In [1]:
from pathlib import Path
import duckdb

workspace_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'internship-warehouse').exists()), Path.cwd())
root = workspace_root / 'data' / 'internship-warehouse'
march = (root / 'fact_content_daily_performance' / 'month=2026-03' / '*.parquet').as_posix()
con = duckdb.connect()
numbers = con.sql(f"""
SELECT
  COUNT(*) AS daily_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_gsc_rows,
  COUNT(DISTINCT client_hash_id || ':' || content_hash_id) FILTER (WHERE gsc_data_available IS TRUE) AS client_content_rows,
  MIN(report_date) AS first_date,
  MAX(report_date) AS last_date
FROM read_parquet('{march}')
""").df()
print(f"March daily rows: {numbers.loc[0, 'daily_rows']:,}")
print(f"Rows with gsc_data_available IS TRUE: {numbers.loc[0, 'available_gsc_rows']:,}")
print(f"Available client-content groups: {numbers.loc[0, 'client_content_rows']:,}")
print(f"Date span: {numbers.loc[0, 'first_date']} to {numbers.loc[0, 'last_date']}")
numbers

March daily rows: 9,841,378
Rows with gsc_data_available IS TRUE: 3,611,061
Available client-content groups: 176,738
Date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,daily_rows,available_gsc_rows,client_content_rows,first_date,last_date
0,9841378,3611061,176738,2026-03-01,2026-03-31


## 3) Why data or ML can help

A fixed CTR threshold ignores that expected CTR changes with position, impressions, page type, intent, and seasonality. Data can make those differences visible. A simple rule is an appropriate first baseline; ML is useful only if it learns a ranking that beats that rule on a future window and an honest client- or time-aware split. The value is better prioritization of scarce review time, not automation for its own sake.

## 4) Careful words

I can claim that the March slice contains substantial observable Search Console coverage and that CTR opportunity can be measured as a prioritization signal. I cannot claim that a low CTR is caused by weak metadata, that a recommendation will increase clicks, or that this one month generalizes to every client. Those claims require future-window outcomes, grouped or time-aware validation, and human review.

## 5) Self-check

- Lane chosen: CTR / Engagement Opportunity Scoring.
- Decision and action named: prioritize pages for metadata or content review.
- Unit named: client-content-month.
- Cost of a wrong call named.
- At least two real March warehouse numbers are printed.
- The framing distinguishes prioritization evidence from causal or production claims.